# FS1 Final-Shot — 36 Multimodal Evidence + Query-Local Event Graph

Exactly three arms: B0 (frozen BCF-1 F1), M0 (multimodal/no graph), M1 (same evidence + one graph revision). Predictions for both benchmarks are finalized and hashed before GT is opened. KIS/TRAKE retain exact B0 Top5; QA does not. No production promotion.

In [ ]:
import os
from pathlib import Path
REPO_URL = "https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git"
REPO_REF = "TRIAGEEG"
ANCHOR = "56c2f37df6841af0e7fe858632ccf8554e8ac4e1"
REPO_DIR = Path(os.environ.get("AIC_REPO_DIR", "/kaggle/working/AIC2026_TeamPTK_SGU"))
RAW_INPUT = Path(os.environ.get("AIC_DATA_ROOT", "/kaggle/input/datasets/nadkli/dataset-aic"))
FREEZE_INPUT = Path(os.environ.get("AIC_FS1_FREEZE_ROOT", "/kaggle/input/datasets/irthn1311/fs1-master-preparation-freeze-2026-08-18"))
TEAM_EVAL_INPUT=Path(os.environ.get("AIC_TEAM_EVAL_ROOT","/kaggle/input/datasets/irthn1311/aic2026_team_eval_dev_v1"))
EVIDENCE_INPUT=Path(os.environ.get("AIC_FS1_EVIDENCE_ROOT","/kaggle/input/datasets/irthn1311/triage-eg-fs1-assets-evidence-v01"))
QWEN_INPUT=Path(os.environ.get("AIC_QWEN_ASSET_ROOT","/kaggle/input/datasets/irthn1311/fs1-qwen2-5-vl-3b-instruct"))
OUTPUT_ROOT=Path("/kaggle/working/triage_eg_fs1_finalshot_v01")
OUTPUT_ZIP=Path("/kaggle/working/triage_eg_fs1_finalshot_v01_bundle.zip")
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
print({"required_inputs":{"raw_dataset":str(RAW_INPUT),"team_eval":str(TEAM_EVAL_INPUT),"fs1_freeze":str(FREEZE_INPUT),"fs1_evidence":str(EVIDENCE_INPUT),"qwen_asset":str(QWEN_INPUT)},"internet_required":"ONLY_FOR_GIT_CLONE","model_download_required":False,"output_zip":str(OUTPUT_ZIP)})


In [ ]:
import subprocess, sys
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git","clone","--branch",REPO_REF,"--single-branch",REPO_URL,str(REPO_DIR)],check=True)
subprocess.run(["git","fetch","origin",REPO_REF],cwd=REPO_DIR,check=True)
subprocess.run(["git","checkout","--detach","FETCH_HEAD"],cwd=REPO_DIR,check=True)
HEAD=subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO_DIR,text=True).strip()
ancestor=subprocess.run(["git","merge-base","--is-ancestor",ANCHOR,HEAD],cwd=REPO_DIR).returncode==0
if not ancestor: raise RuntimeError(f"FS1 lineage mismatch: anchor={ANCHOR} HEAD={HEAD}")
sys.path.insert(0,str(REPO_DIR/"src"))
print({"source_ref":REPO_REF,"HEAD":HEAD,"checkout_mode":"DETACHED_FETCH_HEAD","anchor_is_ancestor":ancestor})

# Fail immediately before expensive work when the checked-out source lacks FS1.
required_fs1 = REPO_DIR / "src/triage_eg/fs1/runner.py"
if not required_fs1.is_file():
    raise RuntimeError(
        "FS1_SOURCE_NOT_AVAILABLE_AT_CHECKED_OUT_REF: Notebook 36 requires the "
        "implemented src/triage_eg/fs1 package to be committed on TRIAGEEG while "
        "56c2f37 remains an ancestor. Do not run expensive preprocessing first."
    )


In [ ]:
import hashlib,json,shutil,zipfile
def resolve(root,name):
    matches=sorted(root.rglob(name)) if root.exists() else []
    if len(matches)!=1: raise RuntimeError(f"Expected exactly one {name} under {root}; found {matches}")
    return matches[0]
FREEZE_ZIP=resolve(FREEZE_INPUT,"FS1_MASTER_PREPARATION_FREEZE_2026-08-18.zip") if FREEZE_INPUT.is_dir() and not (FREEZE_INPUT/"fs1_master_preparation").is_dir() else None
FREEZE_ROOT=Path("/kaggle/working/fs1_master_freeze")
if FREEZE_ZIP:
    with zipfile.ZipFile(FREEZE_ZIP) as archive: archive.extractall(FREEZE_ROOT)
else: FREEZE_ROOT=FREEZE_INPUT
PREP=next(FREEZE_ROOT.rglob("FS1_PROTOCOL.md")).parent
EVIDENCE_MANIFEST=resolve(EVIDENCE_INPUT,"evidence_manifest.json")
PLUGIN_STATUS=json.loads(resolve(EVIDENCE_INPUT,"plugin_status.json").read_text())
print({"freeze":str(PREP),"evidence_manifest":str(EVIDENCE_MANIFEST)})


In [ ]:
# Required tests run before any benchmark prediction.
test=subprocess.run([sys.executable,"-m","pytest","tests/unit/fs1","tests/unit/bcf1_protected_late_fusion","tests/unit/sca1_siglip2_complementarity","-q"],cwd=REPO_DIR,capture_output=True,text=True)
TEST_SUMMARY={"returncode":test.returncode,"tail":test.stdout.splitlines()[-20:]}
if test.returncode: raise RuntimeError(TEST_SUMMARY)
print(TEST_SUMMARY)


In [ ]:
from triage_eg.fs1.io import PreGTGate,read_jsonl,write_jsonl,sha256
from triage_eg.fs1.runner import build_arm
CROSS_B0=PREP/"frozen_baseline/cross_b0_bcf1_f1.jsonl"; L21_B0=PREP/"frozen_baseline/l21_b0_bcf1_f1.jsonl"
expected={"cross":"801e9e4a8e33916cb0430c9c391694410972a84b212d0db949d63671be39e2dc","l21":"3c4dbd2bf4766b286d1efceded120c801e59696d08ab3deb19dd38669074fd16"}
if sha256(CROSS_B0)!=expected["cross"] or sha256(L21_B0)!=expected["l21"]: raise RuntimeError("FS1 frozen B0 hash mismatch")
def bench_root(name):
    matches=sorted(p.parent for p in TEAM_EVAL_INPUT.rglob("queries.jsonl") if p.parent.name==name)
    if len(matches)!=1: raise RuntimeError(f"benchmark discovery failed {name}: {matches}")
    return matches[0]
BENCH={"cross":bench_root("dev_cross_60"),"l21":bench_root("dev_l21_150")}
QUERY_ONLY={}
for name,root in BENCH.items():
    target=Path("/kaggle/working/fs1_query_only")/name; target.mkdir(parents=True,exist_ok=True)
    shutil.copy2(root/"queries.jsonl",target/"queries.jsonl")
    assert {p.name for p in target.iterdir()}=={"queries.jsonl"}; QUERY_ONLY[name]=target


In [ ]:
# Load only precomputed, non-GT evidence records.
queries={name:read_jsonl(root/"queries.jsonl") for name,root in QUERY_ONLY.items()}
b0={"cross":read_jsonl(CROSS_B0),"l21":read_jsonl(L21_B0)}
evidence={name:{} for name in BENCH}
for benchmark in BENCH:
    for modality in ("asr","ocr","action","object"):
        matches=list(EVIDENCE_INPUT.rglob(f"{benchmark}_{modality}_evidence.jsonl"))
        grouped={}
        if len(matches)==1:
            for row in read_jsonl(matches[0]): grouped.setdefault(str(row["query_id"]),[]).append(row)
        evidence[benchmark][modality]=grouped
    lexical=json.loads(resolve(EVIDENCE_INPUT,"asr_lexical_index.json").read_text())
    b0_grouped={}
    for row in b0[benchmark]: b0_grouped.setdefault(str(row["query_id"]),[]).append(row)
    asr_grouped={}
    import re
    for query in queries[benchmark]:
        query_id=str(query["query_id"]); text=" ".join(str(query.get(k,"")) for k in ("query","description","question"))
        videos={hit["video_id"] for token in set(re.findall(r"\w+",text.casefold())) for hit in lexical.get(token,[])}
        rows=[{**row,"source":"asr"} for row in b0_grouped[query_id] if row.get("video_id") in videos]
        asr_grouped[query_id]=rows
    evidence[benchmark]["asr"]=asr_grouped
enabled={"asr":True,"ocr":PLUGIN_STATUS.get("ppocr",{}).get("enabled",False),"action":PLUGIN_STATUS.get("xclip",{}).get("enabled",False),"object":PLUGIN_STATUS.get("object",{}).get("enabled",False)}


In [ ]:
# Mandatory Qwen QA branch: bounded top-20, IDs inherited only from B0 candidates.
from triage_eg.data.stage0_audit.asset_resolver import discover_layout,resolve_assets
from triage_eg.fs1.qa import GroundingCandidate,bounded_grounding_candidates
from triage_eg.fs1.qwen_adapter import QwenEvidenceAdapter
from triage_eg.fs1.runner import group_predictions
from triage_eg.video import OpenCVRawVideoDecoder
qwen_roots=sorted(p.parent for p in QWEN_INPUT.rglob("config.json") if p.is_file())
if len(qwen_roots)!=1: raise RuntimeError(f"Qwen asset discovery failed: {qwen_roots}")
adapter=QwenEvidenceAdapter(qwen_roots[0]); adapter.load()
video_parts,keyframe_parts=discover_layout(RAW_INPUT); qwen_audit=[]
for benchmark in ("cross","l21"):
    grouped=group_predictions(b0[benchmark]); qwen_by_query={}
    for query in queries[benchmark]:
        if str(query["task"]).upper()!="QA": continue
        candidates=bounded_grounding_candidates([GroundingCandidate(str(row["video_id"]),int(row["frame_id"]),int(row["rank"]),{"source":"B0"}) for row in grouped[str(query["query_id"])]])
        answers=[]
        for candidate in candidates:
            try:
                assets=resolve_assets(RAW_INPUT,candidate.video_id,video_parts,keyframe_parts)
                decoder=OpenCVRawVideoDecoder(candidate.video_id,assets.video)
                frame=decoder.decode_indices([candidate.frame_id])[0]; decoder.close()
                parsed,audit=adapter.answer(candidate,frame.image,description=str(query.get("description",query.get("query",""))),question=str(query.get("question","")))
                if parsed and parsed["evidence_sufficient"]: answers.append({**parsed,"query_id":str(query["query_id"]),"rank":candidate.evidence_rank,"source":"qwen"})
                qwen_audit.append({"benchmark":benchmark,"query_id":str(query["query_id"]),**audit})
            except Exception as error: qwen_audit.append({"benchmark":benchmark,"query_id":str(query["query_id"]),"candidate":candidate.__dict__,"status":"FALLBACK","error":f"{type(error).__name__}: {error}"})
        qwen_by_query[str(query["query_id"])]=answers
    evidence[benchmark]["qwen"]=qwen_by_query
adapter.unload()
(OUTPUT_ROOT/"qwen_diagnostics.jsonl").write_text("".join(json.dumps(x,ensure_ascii=False,default=str)+"\n" for x in qwen_audit))


In [ ]:
# PRE-GT: B0 -> hash, M0 -> hash, M1 -> hash for both benchmarks.
gate=PreGTGate(); prediction_paths={}; diagnostics=[]
for benchmark in ("cross","l21"):
    b0_path=OUTPUT_ROOT/f"predictions/{benchmark}_B0.jsonl"; b0_path.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(CROSS_B0 if benchmark=="cross" else L21_B0,b0_path)
    prediction_paths[(benchmark,"B0")]=b0_path; gate.finalize(benchmark,"B0",b0_path)
    for arm in ("M0","M1"):
        rows,diag=build_arm(arm,queries[benchmark],b0[benchmark],evidence[benchmark],enabled)
        path=OUTPUT_ROOT/f"predictions/{benchmark}_{arm}.jsonl"; write_jsonl(path,rows)
        prediction_paths[(benchmark,arm)]=path; gate.finalize(benchmark,arm,path); diagnostics.extend(diag)
(OUTPUT_ROOT/"prediction_hashes.json").write_text(json.dumps(gate.hashes,indent=2)+"\n")
(OUTPUT_ROOT/"routing_graph_diagnostics.jsonl").write_text("".join(json.dumps(x,ensure_ascii=False)+"\n" for x in diagnostics))
print(gate.hashes)


In [ ]:
# ONLY NOW may GT be opened and shared evaluator imported.
gate.open_gt()
from aic2026_eval.io import read_jsonl as eval_read_jsonl
from aic2026_eval.scoring import evaluate
evaluations={}
for benchmark,root in BENCH.items():
    gt=eval_read_jsonl(root/"gt.jsonl"); query_rows=eval_read_jsonl(root/"queries.jsonl")
    evaluations[benchmark]={}
    for arm in ("B0","M0","M1"):
        aggregate,per_query,slices,issues=evaluate(query_rows,eval_read_jsonl(prediction_paths[(benchmark,arm)]),gt)
        evaluations[benchmark][arm]={**aggregate,"tasks":{task:slices[f"task:{task}"] for task in ("KIS","QA","TRAKE")},"per_query":per_query,"issues":issues}


In [ ]:
from triage_eg.fs1.selection import select_arm
# Normalize evaluator payloads to the frozen selector contract.
metrics={arm:{"cross":evaluations["cross"][arm],"l21":evaluations["l21"][arm]} for arm in ("B0","M0","M1")}
integrity={"M0":True,"M1":True}
DECISION=select_arm(metrics,integrity)
(OUTPUT_ROOT/"official_evaluations.json").write_text(json.dumps(evaluations,indent=2,default=str)+"\n")
(OUTPUT_ROOT/"decision.json").write_text(json.dumps(DECISION,indent=2)+"\n")
print(DECISION)


In [ ]:
# Formal report and required compact manifests.
manifest={"HEAD":HEAD,"anchor":ANCHOR,"arms":["B0","M0","M1"],"gt_opened_after_all_hashes":gate.gt_opened,"production_policy_changed":False,"qa_alias_matching_is_official_btc_semantics":False}
for name,value in (("run_manifest.json",manifest),("asset_manifest.json",json.loads(resolve(EVIDENCE_INPUT,"asset_manifest.json").read_text())),("evidence_manifest.json",json.loads(EVIDENCE_MANIFEST.read_text())),("plugin_status.json",PLUGIN_STATUS),("tests_summary.json",TEST_SUMMARY)): (OUTPUT_ROOT/name).write_text(json.dumps(value,indent=2,default=str)+"\n")
report=["# TRIAGE-EG FS1 Final-Shot v0.1",f"Selected arm: {DECISION['selected_arm']}","","Exactly B0/M0/M1 were evaluated. All prediction hashes were finalized before GT.","KIS/TRAKE B0 Top5 protection and QA no-protection are enforced by tests.","QA absolute metrics remain an internal proxy because official BTC alias semantics are unavailable.","Production defaults were not changed."]
(OUTPUT_ROOT/"FORMAL_REPORT.md").write_text("\n".join(report)+"\n")
shutil.make_archive(str(OUTPUT_ZIP.with_suffix("")),"zip",OUTPUT_ROOT)
print({"download_zip":str(OUTPUT_ZIP),"selected":DECISION["selected_arm"]})
